In [106]:
import os
import json
import pandas as pd


def preprocess(file_path, delimiter=",", output_dir=None):
    if output_dir is None:
        output_dir = "preprocess"

    os.makedirs(output_dir, exist_ok=True)

    link_stream = pd.read_csv(
        file_path,
        delimiter=delimiter,
        usecols=[0, 1, 2]
    )
    link_stream.columns = ["source", "destination", "timestamp"]

    nodes = pd.concat([link_stream["source"], link_stream["destination"]]).unique()
    print(f"{len(nodes)} nodes in the link stream")

    node2id = {int(node): int(idx) for idx, node in enumerate(nodes)}

    link_stream["source"] = link_stream["source"].map(node2id)
    link_stream["destination"] = link_stream["destination"].map(node2id)
    link_stream["idx"] = range(len(link_stream))

    t = pd.to_numeric(link_stream["timestamp"], errors="coerce")
    if t.isna().any():
        raise ValueError(f"{file_path} contains non-numeric timestamps.")

    base_name = os.path.splitext(os.path.basename(file_path))[0]

    output_csv = os.path.join(output_dir, base_name + ".csv")
    output_map_csv = os.path.join(output_dir, base_name + "_node2id.csv")

    link_stream.to_csv(output_csv, index=False)

    map_df = pd.DataFrame({
        "node": list(node2id.keys()),
        "id": list(node2id.values())
    })
    map_df.to_csv(output_map_csv, index=False)

    return node2id, len(nodes)

node2id, num_nodes = preprocess(
        file_path="/Users/acw721/Desktop/research/linkstream/syn_data/p0.8_mu0.2_l10_1.csv",
        delimiter=","
    )

50 nodes in the link stream


In [107]:
import os
import numpy as np
import pandas as pd
import networkx as nx
from CTDNE.ctdne import CTDNE


def run_ctdne_to_npy(
    file_path,
    dimensions=64,
    walk_length=30,
    num_walks=200,
    workers=4,
    window=10,
    min_count=1,
    batch_words=4,
):
    df = pd.read_csv(file_path)
    df = df.sort_values("timestamp").reset_index(drop=True)

    G = nx.MultiGraph()
    for row in df.itertuples(index=False):
        u = int(row.source)
        v = int(row.destination)
        t = row.timestamp
        G.add_edge(u, v, time=float(t))

    ctdne_model = CTDNE(
        G,
        dimensions=dimensions,
        walk_length=walk_length,
        num_walks=num_walks,
        workers=workers
    )

    model = ctdne_model.fit(
        window=window,
        min_count=min_count,
        batch_words=batch_words
    )

    nodes_str = model.wv.index_to_key
    emb = model.wv.vectors.astype(np.float32)

    nodes = []
    for x in nodes_str:
        try:
            nodes.append(int(x))
        except:
            raise ValueError(f"Node id {x} cannot be converted to int.")

    max_node_id = int(max(df["source"].max(), df["destination"].max()))
    num_nodes = max_node_id + 1
    dim = emb.shape[1]

    node_features = np.zeros((num_nodes, dim), dtype=np.float32)

    for i, node in enumerate(nodes):
        if 0 <= node < num_nodes:
            node_features[node] = emb[i]

    base_name = os.path.splitext(os.path.basename(file_path))[0]
    out_path = os.path.join(base_name + ".npy")

    np.save(out_path, node_features)

    print(f"Saved embeddings to: {out_path}")
    print(f"node_features.shape = {node_features.shape}")

    return node_features


_ = run_ctdne_to_npy("/Users/acw721/Desktop/research/linkstream/syn_data/p0.8_mu0.2_l10_1.csv")

Generating walks (CPU: 2): 100%|██████████| 50/50 [00:00<00:00, 136.73it/s]


Saved embeddings to: p0.8_mu0.2_l10_1.npy
node_features.shape = (50, 64)


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import networkx as nx
from CTDNE.ctdne import CTDNE

INPUT_DIR = "../syn_data"
OUTPUT_DIR = "../pretrain"

dimensions = 64
walk_length = 30
num_walks = 200
workers = 4
window = 10
min_count = 1
batch_words = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))
print(f"Found {len(csv_files)} csv files.")

for file_path in csv_files:
    print(f"\nProcessing: {file_path}")

    df = pd.read_csv(file_path)
    df = df.sort_values("timestamp").reset_index(drop=True)

    G = nx.MultiGraph()
    for row in df.itertuples(index=False):
        u = int(row.source)
        v = int(row.destination)
        t = float(row.timestamp)
        G.add_edge(u, v, time=t)

    ctdne_model = CTDNE(
        G,
        dimensions=dimensions,
        walk_length=walk_length,
        num_walks=num_walks,
        workers=workers
    )

    model = ctdne_model.fit(
        window=window,
        min_count=min_count,
        batch_words=batch_words
    )

    nodes_str = model.wv.index_to_key
    emb = model.wv.vectors.astype(np.float32)

    nodes = np.array([int(x) for x in nodes_str], dtype=np.int64)
    order = np.argsort(nodes)
    emb = emb[order]

    base = os.path.splitext(os.path.basename(file_path))[0]
    out_path = os.path.join(OUTPUT_DIR, base + ".npy")

    np.save(out_path, emb)
    print(f"Saved to: {out_path}, shape = {emb.shape}")

print("\nAll files done.")

In [83]:
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd


# =========================================================
# 1. 工具函数
# =========================================================
def js_divergence_from_weight_dict(
    left: Dict[Any, float],
    right: Dict[Any, float],
    eps: float = 1e-12,
) -> float:
    """
    计算两个离散分布的 Jensen-Shannon Divergence.
    输入是两个 {neighbor: weight} 字典，内部会自动归一化。
    返回值范围大约在 [0, ln(2)].
    """
    keys = set(left.keys()) | set(right.keys())
    if len(keys) == 0:
        return 0.0

    p = np.array([left.get(k, 0.0) for k in keys], dtype=np.float64)
    q = np.array([right.get(k, 0.0) for k in keys], dtype=np.float64)

    p_sum = p.sum()
    q_sum = q.sum()
    if p_sum <= eps or q_sum <= eps:
        return 0.0

    p = p / p_sum
    q = q / q_sum
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > eps
        return np.sum(a[mask] * np.log((a[mask] + eps) / (b[mask] + eps)))

    jsd = 0.5 * kl(p, m) + 0.5 * kl(q, m)
    return float(jsd)


def choose_beta_from_median_dt(
    timestamps: np.ndarray,
    half_life: float = 1.0,
    eps: float = 1e-12,
) -> float:
    """
    给一个时间序列，按相邻时间差的中位数自动选 beta。
    若希望“中位时间差对应的权重 = exp(-half_life)”，则 beta = half_life / median_dt。
    更常见地，若希望中位时间差是半衰期，可传 half_life = ln(2).
    """
    if len(timestamps) < 2:
        return 1.0

    dt = np.diff(np.sort(np.asarray(timestamps, dtype=np.float64)))
    dt = dt[dt > 0]
    if len(dt) == 0:
        return 1.0

    med_dt = np.median(dt)
    beta = half_life / (med_dt + eps)
    return float(beta)


# =========================================================
# 2. 数据结构
# =========================================================
@dataclass
class NodeEvent:
    node: Any
    event_idx_global: int
    timestamp: float
    neighbor: Any


# =========================================================
# 3. 为每个节点构造事件序列
# =========================================================
def build_node_event_sequences(df: pd.DataFrame) -> Dict[Any, List[NodeEvent]]:
    """
    输入 df 必须包含:
      source, destination, timestamp
    输出:
      node -> 按时间排序的 NodeEvent 列表
    对于每条边 (u, v, t)，会同时给 u 和 v 各生成一个事件。
    """
    required = {"source", "destination", "timestamp"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df = df.copy()
    df = df.sort_values(["timestamp"]).reset_index(drop=True)

    node2events: Dict[Any, List[NodeEvent]] = {}

    for idx, row in df.iterrows():
        u = row["source"]
        v = row["destination"]
        t = float(row["timestamp"])

        node2events.setdefault(u, []).append(
            NodeEvent(node=u, event_idx_global=idx, timestamp=t, neighbor=v)
        )
        node2events.setdefault(v, []).append(
            NodeEvent(node=v, event_idx_global=idx, timestamp=t, neighbor=u)
        )

    # 每个节点内部再按时间排序一次；若同时间戳，按全局边序稳定排序
    for u in node2events:
        node2events[u].sort(key=lambda x: (x.timestamp, x.event_idx_global))

    return node2events


# =========================================================
# 4. 构造“切点左右两段”的时间衰减邻居画像
# =========================================================
def build_side_profile(
    events: List[NodeEvent],
    start: int,
    end: int,
    anchor_time: float,
    beta: float,
    top_k_neighbors: int | None = None,
) -> Dict[Any, float]:
    """
    对 events[start:end+1] 这一段，围绕 anchor_time 构造邻居画像:
        weight = exp(-beta * |anchor_time - t_event|)
    返回 {neighbor: accumulated_weight}
    可选只保留 top-k 邻居。
    """
    profile: Dict[Any, float] = {}

    for i in range(start, end + 1):
        ev = events[i]
        w = math.exp(-beta * abs(anchor_time - ev.timestamp))
        profile[ev.neighbor] = profile.get(ev.neighbor, 0.0) + w

    if top_k_neighbors is not None and len(profile) > top_k_neighbors:
        items = sorted(profile.items(), key=lambda x: x[1], reverse=True)[:top_k_neighbors]
        profile = dict(items)

    return profile


def compute_split_score_jsd(
    events: List[NodeEvent],
    split_idx: int,
    seg_start: int,
    seg_end: int,
    beta: float,
    top_k_neighbors: int | None = None,
) -> float:
    """
    对当前 segment [seg_start, seg_end]，尝试在 split_idx 处切开:
      左段 [seg_start, split_idx]
      右段 [split_idx+1, seg_end]
    定义 split score = 左右邻居画像的 JSD
    """
    left_anchor = events[split_idx].timestamp
    right_anchor = events[split_idx + 1].timestamp

    left_profile = build_side_profile(
        events, seg_start, split_idx, anchor_time=left_anchor, beta=beta, top_k_neighbors=top_k_neighbors
    )
    right_profile = build_side_profile(
        events, split_idx + 1, seg_end, anchor_time=right_anchor, beta=beta, top_k_neighbors=top_k_neighbors
    )

    score = js_divergence_from_weight_dict(left_profile, right_profile)
    return score


# =========================================================
# 5. 在一个 segment 内寻找最佳切点
# =========================================================
def find_best_split_for_segment(
    events: List[NodeEvent],
    seg_start: int,
    seg_end: int,
    beta: float,
    min_seg_len: int = 5,
    top_k_neighbors: int | None = None,
    coarse_step: int = 1,
) -> Tuple[int | None, float]:
    """
    在 [seg_start, seg_end] 内找最佳切点。
    要求左右长度都 >= min_seg_len。
    返回:
      (best_split_idx, best_score)
    若没有合法切点，则返回 (None, -inf)
    """
    length = seg_end - seg_start + 1
    if length < 2 * min_seg_len:
        return None, float("-inf")

    best_idx = None
    best_score = float("-inf")

    left_min_end = seg_start + min_seg_len - 1
    right_max_start = seg_end - min_seg_len

    for split_idx in range(left_min_end, right_max_start, coarse_step):
        score = compute_split_score_jsd(
            events=events,
            split_idx=split_idx,
            seg_start=seg_start,
            seg_end=seg_end,
            beta=beta,
            top_k_neighbors=top_k_neighbors,
        )
        if score > best_score:
            best_score = score
            best_idx = split_idx

    return best_idx, best_score


# =========================================================
# 6. Top-down recursive segmentation
# =========================================================
def recursive_segment_node_events(
    events: List[NodeEvent],
    beta: float,
    min_seg_len: int = 5,
    jsd_threshold: float = 0.10,
    top_k_neighbors: int | None = None,
    max_segments: int | None = None,
) -> List[Tuple[int, int]]:
    """
    对单个节点的事件序列做 top-down segmentation。
    返回若干段 [(start_idx, end_idx), ...]，按时间顺序。
    """
    if len(events) == 0:
        return []

    segments: List[Tuple[int, int]] = [(0, len(events) - 1)]

    while True:
        if max_segments is not None and len(segments) >= max_segments:
            break

        best_gain = float("-inf")
        best_seg_pos = None
        best_split_idx = None

        for seg_pos, (a, b) in enumerate(segments):
            split_idx, score = find_best_split_for_segment(
                events=events,
                seg_start=a,
                seg_end=b,
                beta=beta,
                min_seg_len=min_seg_len,
                top_k_neighbors=top_k_neighbors,
                coarse_step=1,
            )
            if split_idx is not None and score > best_gain:
                best_gain = score
                best_seg_pos = seg_pos
                best_split_idx = split_idx

        if best_seg_pos is None or best_split_idx is None:
            break

        if best_gain < jsd_threshold:
            break

        a, b = segments[best_seg_pos]
        left = (a, best_split_idx)
        right = (best_split_idx + 1, b)

        segments = segments[:best_seg_pos] + [left, right] + segments[best_seg_pos + 1 :]

    return segments


# =========================================================
# 7. 主函数：对整个图做 segmentation
# =========================================================
def segment_graph_by_neighbor_jsd(
    df: pd.DataFrame,
    beta: float | None = None,
    min_seg_len: int = 5,
    jsd_threshold: float = 0.10,
    top_k_neighbors: int | None = 20,
    max_segments_per_node: int | None = None,
) -> Tuple[pd.DataFrame, Dict[Any, List[Tuple[int, int]]], Dict[Any, List[NodeEvent]]]:
    """
    输入:
      df: 包含 source, destination, timestamp 的 DataFrame

    输出:
      result_df:
         在原始边表基础上增加:
           source_segment
           destination_segment

      node_segments:
         node -> [(start_idx, end_idx), ...]

      node2events:
         node -> List[NodeEvent]
    """
    df = df.copy().reset_index(drop=True)

    node2events = build_node_event_sequences(df)

    if beta is None:
        beta = choose_beta_from_median_dt(df["timestamp"].to_numpy(), half_life=np.log(2))
        print(f"[Auto beta] beta = {beta:.6f}")

    node_segments: Dict[Any, List[Tuple[int, int]]] = {}
    node_event_to_segment: Dict[Tuple[Any, int], int] = {}

    # 每个节点独立做分段
    for u, events in node2events.items():
        segs = recursive_segment_node_events(
            events=events,
            beta=beta,
            min_seg_len=min_seg_len,
            jsd_threshold=jsd_threshold,
            top_k_neighbors=top_k_neighbors,
            max_segments=max_segments_per_node,
        )
        node_segments[u] = segs

        for seg_id, (a, b) in enumerate(segs):
            for local_idx in range(a, b + 1):
                global_eid = events[local_idx].event_idx_global
                node_event_to_segment[(u, global_eid)] = seg_id

    # 映射回原始边表
    source_segments = []
    destination_segments = []

    for eid, row in df.iterrows():
        u = row["source"]
        v = row["destination"]

        src_seg = node_event_to_segment.get((u, eid), -1)
        dst_seg = node_event_to_segment.get((v, eid), -1)

        source_segments.append(src_seg)
        destination_segments.append(dst_seg)

    result_df = df.copy()
    result_df["source_segment"] = source_segments
    result_df["destination_segment"] = destination_segments

    return result_df, node_segments, node2events


def load_id_to_node_map(node2id_path: str) -> Dict[Any, Any]:
    node2id_df = pd.read_csv(node2id_path)

    if {"node", "id"}.issubset(node2id_df.columns):
        return dict(zip(node2id_df["id"], node2id_df["node"]))

    elif node2id_df.shape[1] >= 2:
        col0, col1 = node2id_df.columns[:2]

        if "id" in col0.lower():
            id_col, node_col = col0, col1
        elif "id" in col1.lower():
            id_col, node_col = col1, col0
        else:
            # 默认第一列 node，第二列 id
            node_col, id_col = col0, col1

        return dict(zip(node2id_df[id_col], node2id_df[node_col]))

    else:
        raise ValueError(f"Unexpected node2id format: {node2id_path}")
    
if __name__ == "__main__":
    file_path = "../preprocess/p0.8_mu0.2_l20.csv"
    node2id_path = "../preprocess/p0.8_mu0.2_l20_node2id.csv"
    out_path = "segmented_by_neighbor_jsd_l20.csv"

    df = pd.read_csv(file_path)

    result_df, node_segments, node2events = segment_graph_by_neighbor_jsd(
        df=df,
        beta=None,
        min_seg_len=5,
        jsd_threshold=0.1,
        top_k_neighbors=20,
        max_segments_per_node=None
    )

    # 读取 id -> 原始 node 映射
    id_to_node = load_id_to_node_map(node2id_path)

    # 把内部 id 映射回原始 node
    result_df["source"] = result_df["source"].map(id_to_node)
    result_df["destination"] = result_df["destination"].map(id_to_node)

    # 检查是否有没映射上的
    if result_df["source"].isna().any() or result_df["destination"].isna().any():
        bad_src = result_df.loc[result_df["source"].isna(), "source"].unique().tolist()
        bad_dst = result_df.loc[result_df["destination"].isna(), "destination"].unique().tolist()
        raise ValueError(
            "Some node ids cannot be mapped back.\n"
            f"Bad source ids: {bad_src[:10]}\n"
            f"Bad destination ids: {bad_dst[:10]}"
        )

    result_df.to_csv(out_path, index=False)

    print(f"Saved to: {out_path}")
    print(result_df.head())

    print("\nExample node segments:")
    shown = 0
    for u, segs in node_segments.items():
        print(f"node={u}, num_events={len(node2events[u])}, segments={segs}")
        shown += 1
        if shown >= 5:
            break

[Auto beta] beta = 0.115525
Saved to: segmented_by_neighbor_jsd_l20.csv
   source  destination  timestamp  idx  source_segment  destination_segment
0       0           37          6    0               0                    0
1       5           48          9    1               0                    0
2       7           28         16    2               0                    0
3       0           40         20    3               0                    0
4      10           12         24    4               0                    0

Example node segments:
node=0, num_events=59, segments=[(0, 14), (15, 32), (33, 43), (44, 58)]
node=17, num_events=75, segments=[(0, 11), (12, 28), (29, 42), (43, 52), (53, 63), (64, 74)]
node=1, num_events=84, segments=[(0, 18), (19, 29), (30, 39), (40, 55), (56, 69), (70, 83)]
node=48, num_events=76, segments=[(0, 9), (10, 23), (24, 36), (37, 49), (50, 60), (61, 75)]
node=2, num_events=87, segments=[(0, 15), (16, 32), (33, 44), (45, 54), (55, 68), (69, 86)]


## cusum

In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd


# =========================================================
# 1. 工具函数
# =========================================================
def js_divergence_from_weight_dict(
    left: Dict[Any, float],
    right: Dict[Any, float],
    eps: float = 1e-12,
) -> float:
    """
    计算两个离散分布的 Jensen-Shannon Divergence.
    输入是两个 {neighbor: weight} 字典，内部自动归一化。
    返回值范围大约在 [0, ln(2)].
    """
    keys = set(left.keys()) | set(right.keys())
    if len(keys) == 0:
        return 0.0

    p = np.array([left.get(k, 0.0) for k in keys], dtype=np.float64)
    q = np.array([right.get(k, 0.0) for k in keys], dtype=np.float64)

    p_sum = p.sum()
    q_sum = q.sum()
    if p_sum <= eps or q_sum <= eps:
        return 0.0

    p = p / p_sum
    q = q / q_sum
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > eps
        return np.sum(a[mask] * np.log((a[mask] + eps) / (b[mask] + eps)))

    jsd = 0.5 * kl(p, m) + 0.5 * kl(q, m)
    return float(jsd)


def choose_beta_from_median_dt(
    timestamps: np.ndarray,
    half_life: float = None,
    eps: float = 1e-12,
) -> float:
    """
    根据时间差中位数自动选 beta。
    如果 half_life=None，则默认使用 ln(2)，即中位时间差对应半衰期。
    """
    if half_life is None:
        half_life = np.log(2)

    if len(timestamps) < 2:
        return 1.0

    dt = np.diff(np.sort(np.asarray(timestamps, dtype=np.float64)))
    dt = dt[dt > 0]
    if len(dt) == 0:
        return 1.0

    med_dt = np.median(dt)
    beta = half_life / (med_dt + eps)
    return float(beta)


def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    if window <= 1 or len(x) == 0:
        return x.copy()

    out = np.zeros_like(x, dtype=np.float64)
    half = window // 2
    for i in range(len(x)):
        l = max(0, i - half)
        r = min(len(x), i + half + 1)
        out[i] = x[l:r].mean()
    return out


def robust_mean_std(x: np.ndarray, eps: float = 1e-12) -> Tuple[float, float]:
    """
    用 median + MAD 得到更稳的中心和尺度。
    """
    if len(x) == 0:
        return 0.0, 1.0

    med = float(np.median(x))
    mad = float(np.median(np.abs(x - med)))
    std = 1.4826 * mad
    if std < eps:
        std = float(np.std(x))
    if std < eps:
        std = 1.0
    return med, std


def load_id_to_node_map(node2id_path: str) -> Dict[Any, Any]:
    node2id_df = pd.read_csv(node2id_path)

    if {"node", "id"}.issubset(node2id_df.columns):
        return dict(zip(node2id_df["id"], node2id_df["node"]))

    elif node2id_df.shape[1] >= 2:
        col0, col1 = node2id_df.columns[:2]

        if "id" in col0.lower():
            id_col, node_col = col0, col1
        elif "id" in col1.lower():
            id_col, node_col = col1, col0
        else:
            node_col, id_col = col0, col1

        return dict(zip(node2id_df[id_col], node2id_df[node_col]))

    else:
        raise ValueError(f"Unexpected node2id format: {node2id_path}")


# =========================================================
# 2. 数据结构
# =========================================================
@dataclass
class NodeEvent:
    node: Any
    event_idx_global: int
    timestamp: float
    neighbor: Any


# =========================================================
# 3. 为每个节点构造事件序列
# =========================================================
def build_node_event_sequences(df: pd.DataFrame) -> Dict[Any, List[NodeEvent]]:
    required = {"source", "destination", "timestamp"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df = df.copy()
    df = df.sort_values(["timestamp"]).reset_index(drop=True)

    node2events: Dict[Any, List[NodeEvent]] = {}

    for idx, row in df.iterrows():
        u = row["source"]
        v = row["destination"]
        t = float(row["timestamp"])

        node2events.setdefault(u, []).append(
            NodeEvent(node=u, event_idx_global=idx, timestamp=t, neighbor=v)
        )
        node2events.setdefault(v, []).append(
            NodeEvent(node=v, event_idx_global=idx, timestamp=t, neighbor=u)
        )

    for u in node2events:
        node2events[u].sort(key=lambda x: (x.timestamp, x.event_idx_global))

    return node2events


# =========================================================
# 4. 为每个 node-event 构造双边局部邻居分布
# =========================================================
def build_event_profile_bidirectional_k(
    events: List[NodeEvent],
    center_idx: int,
    k_recent: int,
    beta: float,
    top_k_neighbors: int | None = None,
) -> Dict[Any, float]:
    """
    对 events[center_idx] 这个 node-event，
    用它时间上最近的双边 k 个事件构造邻居分布。

    分布定义：
      weight = exp(-beta * abs(t_center - t_j))

    说明：
      - 会包含 center_idx 自己
      - 会从左右两边选时间上最接近的事件
      - 总共最多选 k_recent 个事件
    """
    n = len(events)
    t_center = events[center_idx].timestamp

    # 收集每个事件到中心事件的时间距离
    dist_pairs = []
    for j in range(n):
        dt = abs(events[j].timestamp - t_center)
        # 用 (dt, |j-center_idx|, j) 保证排序稳定，优先选时间更近、索引更近的事件
        dist_pairs.append((dt, abs(j - center_idx), j))

    dist_pairs.sort(key=lambda x: (x[0], x[1], x[2]))
    chosen = [j for _, _, j in dist_pairs[:k_recent]]

    profile: Dict[Any, float] = {}
    for j in chosen:
        ev = events[j]
        dt = abs(t_center - ev.timestamp)
        w = math.exp(-beta * dt)
        profile[ev.neighbor] = profile.get(ev.neighbor, 0.0) + w

    if top_k_neighbors is not None and len(profile) > top_k_neighbors:
        items = sorted(profile.items(), key=lambda x: x[1], reverse=True)[:top_k_neighbors]
        profile = dict(items)

    return profile


def build_event_profiles_for_node(
    events: List[NodeEvent],
    k_recent: int,
    beta: float,
    top_k_neighbors: int | None = None,
    min_history: int = 2,
) -> List[Dict[Any, float] | None]:
    """
    为一个节点的所有事件构造双边局部邻居分布。
    历史太短的前几个事件不再特殊处理，只要总事件数足够，就都可以构造。
    但如果整个节点事件数过少(< min_history)，则全部返回 None。
    """
    n = len(events)
    profiles: List[Dict[Any, float] | None] = []

    if n < min_history:
        return [None] * n

    k_eff = min(k_recent, n)
    for i in range(n):
        p = build_event_profile_bidirectional_k(
            events=events,
            center_idx=i,
            k_recent=k_eff,
            beta=beta,
            top_k_neighbors=top_k_neighbors,
        )
        profiles.append(p)

    return profiles


# =========================================================
# 5. 计算相邻事件 JSD 序列
# =========================================================
def compute_adjacent_jsd_sequence(
    profiles: List[Dict[Any, float] | None]
) -> np.ndarray:
    """
    返回长度与 events 相同的序列 jsd_seq:
      jsd_seq[i] 表示 profiles[i-1] 和 profiles[i] 的 JSD
    其中 i=0 位置没有前驱，记为 0
    """
    n = len(profiles)
    jsd_seq = np.zeros(n, dtype=np.float64)

    for i in range(1, n):
        if profiles[i - 1] is None or profiles[i] is None:
            jsd_seq[i] = 0.0
        else:
            jsd_seq[i] = js_divergence_from_weight_dict(profiles[i - 1], profiles[i])

    return jsd_seq


# =========================================================
# 6. CUSUM 检测变化点
# =========================================================
def detect_change_points_cusum(
    score_seq: np.ndarray,
    warmup: int = 5,
    drift_scale: float = 0.5,
    threshold_scale: float = 5.0,
    min_gap: int = 5,
) -> List[int]:
    """
    对 score_seq 做单侧 CUSUM 检测。
    返回变化点索引列表 cp_idx，含义是：
      在 cp_idx 处开始进入一个新 segment
    """
    n = len(score_seq)
    if n <= warmup + 1:
        return []

    baseline_part = score_seq[:warmup]
    mu, sigma = robust_mean_std(baseline_part)

    k = drift_scale * sigma
    h = threshold_scale * sigma

    cps: List[int] = []
    S = 0.0
    last_cp = -10**9

    for i in range(warmup, n):
        x = score_seq[i]
        S = max(0.0, S + (x - mu - k))

        if S > h:
            if i - last_cp >= min_gap:
                cps.append(i)
                last_cp = i
            S = 0.0

    return cps


# =========================================================
# 7. 把变化点转成 segments
# =========================================================
def change_points_to_segments(
    n_events: int,
    cp_indices: List[int],
    min_seg_len: int = 5,
) -> List[Tuple[int, int]]:
    """
    cp_indices 中的每个索引 i 表示：
      events[i] 是新 segment 的起点
    """
    if n_events == 0:
        return []

    cut_points = sorted(set([i for i in cp_indices if 0 < i < n_events]))

    segs: List[Tuple[int, int]] = []
    start = 0

    for cp in cut_points:
        end = cp - 1
        if end - start + 1 < min_seg_len:
            continue
        segs.append((start, end))
        start = cp

    if n_events - start >= min_seg_len:
        segs.append((start, n_events - 1))
    else:
        if len(segs) == 0:
            segs = [(0, n_events - 1)]
        else:
            prev_start, _ = segs[-1]
            segs[-1] = (prev_start, n_events - 1)

    if len(segs) == 0:
        segs = [(0, n_events - 1)]

    return segs


# =========================================================
# 8. 单节点 segmentation：双边 event-profile + JSD + smoothing + CUSUM
# =========================================================
def segment_single_node_by_event_jsd_cusum(
    events: List[NodeEvent],
    beta: float,
    k_recent: int = 10,
    top_k_neighbors: int | None = 20,
    min_history: int = 2,
    smooth_window: int = 5,
    warmup: int = 5,
    drift_scale: float = 0.5,
    threshold_scale: float = 5.0,
    min_gap: int = 5,
    min_seg_len: int = 5,
    max_segments: int | None = None,
) -> Tuple[List[Tuple[int, int]], np.ndarray, np.ndarray]:
    """
    返回：
      segments
      raw_jsd_seq
      smoothed_jsd_seq
    """
    n = len(events)
    if n == 0:
        return [], np.array([]), np.array([])

    profiles = build_event_profiles_for_node(
        events=events,
        k_recent=k_recent,
        beta=beta,
        top_k_neighbors=top_k_neighbors,
        min_history=min_history,
    )

    raw_jsd = compute_adjacent_jsd_sequence(profiles)
    smoothed_jsd = moving_average(raw_jsd, window=smooth_window)

    cps = detect_change_points_cusum(
        score_seq=smoothed_jsd,
        warmup=min(warmup, max(1, n - 1)),
        drift_scale=drift_scale,
        threshold_scale=threshold_scale,
        min_gap=min_gap,
    )

    if max_segments is not None and len(cps) + 1 > max_segments:
        cp_scores = [(cp, smoothed_jsd[cp]) for cp in cps]
        cp_scores = sorted(cp_scores, key=lambda x: x[1], reverse=True)
        cp_scores = cp_scores[:max_segments - 1]
        cps = sorted([cp for cp, _ in cp_scores])

    segs = change_points_to_segments(
        n_events=n,
        cp_indices=cps,
        min_seg_len=min_seg_len,
    )

    return segs, raw_jsd, smoothed_jsd


# =========================================================
# 9. 主函数：对整个图做 segmentation
# =========================================================
def segment_graph_by_event_jsd_cusum(
    df: pd.DataFrame,
    beta: float | None = None,
    k_recent: int = 10,
    top_k_neighbors: int | None = 20,
    min_history: int = 2,
    smooth_window: int = 5,
    warmup: int = 5,
    drift_scale: float = 0.5,
    threshold_scale: float = 5.0,
    min_gap: int = 5,
    min_seg_len: int = 5,
    max_segments_per_node: int | None = None,
) -> Tuple[pd.DataFrame, Dict[Any, List[Tuple[int, int]]], Dict[Any, List[NodeEvent]], Dict[Any, np.ndarray]]:
    """
    输出：
      result_df: 原始边表 + source_segment + destination_segment
      node_segments: node -> [(start_idx, end_idx), ...]
      node2events: node -> 事件序列
      node2smoothed_jsd: node -> 平滑后的 JSD 序列
    """
    df = df.copy().reset_index(drop=True)
    node2events = build_node_event_sequences(df)

    if beta is None:
        beta = choose_beta_from_median_dt(df["timestamp"].to_numpy(), half_life=np.log(2))
        print(f"[Auto beta] beta = {beta:.6f}")

    node_segments: Dict[Any, List[Tuple[int, int]]] = {}
    node_event_to_segment: Dict[Tuple[Any, int], int] = {}
    node2smoothed_jsd: Dict[Any, np.ndarray] = {}

    for u, events in node2events.items():
        segs, raw_jsd, smoothed_jsd = segment_single_node_by_event_jsd_cusum(
            events=events,
            beta=beta,
            k_recent=k_recent,
            top_k_neighbors=top_k_neighbors,
            min_history=min_history,
            smooth_window=smooth_window,
            warmup=warmup,
            drift_scale=drift_scale,
            threshold_scale=threshold_scale,
            min_gap=min_gap,
            min_seg_len=min_seg_len,
            max_segments=max_segments_per_node,
        )

        node_segments[u] = segs
        node2smoothed_jsd[u] = smoothed_jsd

        for seg_id, (a, b) in enumerate(segs):
            for local_idx in range(a, b + 1):
                global_eid = events[local_idx].event_idx_global
                node_event_to_segment[(u, global_eid)] = seg_id

    source_segments = []
    destination_segments = []

    for eid, row in df.iterrows():
        u = row["source"]
        v = row["destination"]

        src_seg = node_event_to_segment.get((u, eid), -1)
        dst_seg = node_event_to_segment.get((v, eid), -1)

        source_segments.append(src_seg)
        destination_segments.append(dst_seg)

    result_df = df.copy()
    result_df["source_segment"] = source_segments
    result_df["destination_segment"] = destination_segments

    return result_df, node_segments, node2events, node2smoothed_jsd


# =========================================================
# 10. 示例入口
# =========================================================
if __name__ == "__main__":
    file_path = "../preprocess/p0.8_mu0.2_l10_1.csv"
    node2id_path = "../preprocess/p0.8_mu0.2_l20_node2id.csv"
    out_path = "segmented_by_event_jsd_cusum_l20.csv"

    df = pd.read_csv(file_path)

    result_df, node_segments, node2events, node2smoothed_jsd = segment_graph_by_event_jsd_cusum(
        df=df,
        beta=None,                  # 自动估计
        k_recent=20,               # 每个 node-event 看时间上最近多少个双边事件
        top_k_neighbors=20,        # 分布中最多保留多少个邻居
        min_history=2,             # 如果整个节点事件数太少，则不构建 profile
        smooth_window=5,           # JSD 序列平滑窗口
        warmup=5,                  # CUSUM 前期基线长度
        drift_scale=0.5,           # CUSUM k = drift_scale * sigma
        threshold_scale=5.0,       # CUSUM h = threshold_scale * sigma
        min_gap=5,                 # 相邻变化点最小间隔
        min_seg_len=5,             # 最短 segment 长度
        max_segments_per_node=None # 可改成 3 / 5 等
    )

    id_to_node = load_id_to_node_map(node2id_path)

    result_df["source"] = result_df["source"].map(id_to_node)
    result_df["destination"] = result_df["destination"].map(id_to_node)

    if result_df["source"].isna().any() or result_df["destination"].isna().any():
        bad_src = result_df.loc[result_df["source"].isna(), "source"].unique().tolist()
        bad_dst = result_df.loc[result_df["destination"].isna(), "destination"].unique().tolist()
        raise ValueError(
            "Some node ids cannot be mapped back.\n"
            f"Bad source ids: {bad_src[:10]}\n"
            f"Bad destination ids: {bad_dst[:10]}"
        )

    result_df.to_csv(out_path, index=False)

    print(f"Saved to: {out_path}")
    print(result_df.head())

    print("\nExample node segments:")
    shown = 0
    for u, segs in node_segments.items():
        print(f"node={u}, num_events={len(node2events[u])}, segments={segs}")
        shown += 1
        if shown >= 5:
            break

[Auto beta] beta = 0.115525
Saved to: segmented_by_event_jsd_cusum_l20.csv
   source  destination  timestamp  idx  source_segment  destination_segment
0       0           37          6    0               0                    0
1       5           48          9    1               0                    0
2       7           28         16    2               0                    0
3       0           40         20    3               0                    0
4      10           12         24    4               0                    0

Example node segments:
node=0, num_events=59, segments=[(0, 58)]
node=17, num_events=75, segments=[(0, 7), (8, 12), (13, 17), (18, 22), (23, 27), (28, 33), (34, 38), (39, 43), (44, 48), (49, 53), (54, 58), (59, 63), (64, 69), (70, 74)]
node=1, num_events=84, segments=[(0, 83)]
node=48, num_events=76, segments=[(0, 75)]
node=2, num_events=87, segments=[(0, 4), (5, 9), (10, 14), (15, 19), (20, 24), (25, 29), (30, 34), (35, 39), (40, 44), (45, 49), (50, 54), (55, 59)

In [101]:
import math
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Any

import networkx as nx
import numpy as np
import pandas as pd
from gensim.models import Word2Vec


# =========================================================
# 1. 基本配置
# =========================================================
FILE_NAME = "p0.8_mu0.2_l20"
INPUT_PATH = "segmented_by_event_jsd_cusum_l20.csv"

NODE_INFO_OUT = "node_segment_node_info_l20.csv"
EMB_NPY_OUT = "node_segment_node2vec_l20.npy"
EMB_MAP_OUT = "node_segment_node2vec_map_l20.csv"

# -------- self-edge 参数 --------
# self-edge 相似度 = exp(-gamma * JSD)
SELF_EDGE_GAMMA = 1.0

# -------- relax rate 参数 --------
# 以 RELAX_RATE 的概率跳到“自己其他 segment”
# 以 1-RELAX_RATE 的概率跳到“其他节点”
RELAX_RATE = 0.9

# -------- node2vec / word2vec 参数 --------
DIMENSIONS = 64
WALK_LENGTH = 50
NUM_WALKS = 100
WINDOW_SIZE = 20
MIN_COUNT = 1
BATCH_WORDS = 4
WORKERS = 4

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# =========================================================
# 2. 工具函数
# =========================================================
def make_token(node: Any, seg: Any) -> str:
    return f"{node}__seg{seg}"


def parse_token(token: str) -> Tuple[str, int]:
    node, seg = token.rsplit("__seg", 1)
    return node, int(seg)


def js_divergence_from_dicts(
    p_dict: Dict[Any, float],
    q_dict: Dict[Any, float],
    eps: float = 1e-12,
) -> float:
    """
    计算两个离散分布的 JSD.
    输入是 {neighbor: weight}，函数内部会归一化成概率分布。
    JSD 取值范围大约在 [0, ln(2)].
    """
    keys = set(p_dict.keys()) | set(q_dict.keys())
    if len(keys) == 0:
        return 0.0

    p = np.array([p_dict.get(k, 0.0) for k in keys], dtype=np.float64)
    q = np.array([q_dict.get(k, 0.0) for k in keys], dtype=np.float64)

    if p.sum() <= eps or q.sum() <= eps:
        return 0.0

    p = p / (p.sum() + eps)
    q = q / (q.sum() + eps)
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > eps
        return np.sum(a[mask] * np.log((a[mask] + eps) / (b[mask] + eps)))

    jsd = 0.5 * kl(p, m) + 0.5 * kl(q, m)
    return float(jsd)


def weighted_choice(items: List[Any], weights: List[float]) -> Any:
    weights = np.asarray(weights, dtype=np.float64)
    s = weights.sum()
    if s <= 0:
        return random.choice(items)
    probs = weights / s
    idx = np.random.choice(len(items), p=probs)
    return items[idx]


# =========================================================
# 3. 读取 segmented 文件
# =========================================================
df = pd.read_csv(INPUT_PATH)

required_cols = {
    "source", "destination", "timestamp",
    "source_segment", "destination_segment"
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in input file: {missing}")

df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Loaded {len(df)} rows from: {INPUT_PATH}")

# 构造 node-segment token
df["source_token"] = df.apply(lambda r: make_token(r["source"], r["source_segment"]), axis=1)
df["destination_token"] = df.apply(lambda r: make_token(r["destination"], r["destination_segment"]), axis=1)


# =========================================================
# 4. interaction edge：node-segment 之间的原始交互次数
# =========================================================
interaction_counter = defaultdict(float)

for row in df.itertuples(index=False):
    a = row.source_token
    b = row.destination_token
    key = tuple(sorted((a, b)))
    interaction_counter[key] += 1.0

interaction_edges = []
for (a, b), w in interaction_counter.items():
    interaction_edges.append({
        "u": a,
        "v": b,
        "weight": float(w),
        "edge_type": "interaction"
    })

interaction_edges_df = pd.DataFrame(interaction_edges)
print(f"Constructed {len(interaction_edges_df)} interaction edges.")


# =========================================================
# 5. 为每个 node-segment 构造“邻居分布”
#    这里的邻居分布是对“原始邻居节点”的分布，不是对 token 的分布
# =========================================================
segment_neighbor_counter = defaultdict(lambda: defaultdict(float))

for row in df.itertuples(index=False):
    segment_neighbor_counter[row.source_token][row.destination] += 1.0
    segment_neighbor_counter[row.destination_token][row.source] += 1.0

# 记录每个 token 的 node / segment
all_tokens = sorted(set(df["source_token"]).union(set(df["destination_token"])))
token_info_rows = []
for tok in all_tokens:
    node, seg = parse_token(tok)
    token_info_rows.append({
        "token": tok,
        "node": node,
        "segment": seg
    })
token_info_df = pd.DataFrame(token_info_rows)
token_info_df.to_csv(NODE_INFO_OUT, index=False)
print(f"Saved node info to: {NODE_INFO_OUT}")


# =========================================================
# 6. self-edge：同一节点相邻 segment 之间的边
#    权重 = exp(-gamma * JSD(neighbor_dist_i, neighbor_dist_{i+1}))
# =========================================================
node_to_segments = defaultdict(list)
for row in token_info_df.itertuples(index=False):
    node_to_segments[row.node].append((row.segment, row.token))

for node in node_to_segments:
    node_to_segments[node].sort(key=lambda x: x[0])

self_edges = []
for node, seg_list in node_to_segments.items():
    if len(seg_list) <= 1:
        continue

    # 只连相邻 segment
    for i in range(len(seg_list) - 1):
        seg_i, tok_i = seg_list[i]
        seg_j, tok_j = seg_list[i + 1]

        dist_i = segment_neighbor_counter[tok_i]
        dist_j = segment_neighbor_counter[tok_j]

        jsd = js_divergence_from_dicts(dist_i, dist_j)
        sim = math.exp(-SELF_EDGE_GAMMA * jsd)

        self_edges.append({
            "u": tok_i,
            "v": tok_j,
            "weight": float(sim),
            "edge_type": "self"
        })

self_edges_df = pd.DataFrame(self_edges)
print(f"Constructed {len(self_edges_df)} self edges.")


# =========================================================
# 7. 合并成静态图边表
# =========================================================
graph_edges_df = pd.concat([interaction_edges_df, self_edges_df], axis=0, ignore_index=True)
print(graph_edges_df.head())
graph_edges_df.to_csv("graph.csv")

# =========================================================
# 8. 构造 networkx 加权无向图
# =========================================================
G = nx.Graph()

for tok in all_tokens:
    G.add_node(tok)

for row in graph_edges_df.itertuples(index=False):
    u = row.u
    v = row.v
    w = float(row.weight)
    etype = row.edge_type

    if G.has_edge(u, v):
        G[u][v]["weight"] += w
        old_type = G[u][v].get("edge_type", "")
        if etype not in old_type.split("+"):
            G[u][v]["edge_type"] = old_type + "+" + etype if old_type else etype
    else:
        G.add_edge(u, v, weight=w, edge_type=etype)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


# =========================================================
# 9. relax-rate 风格随机游走
# =========================================================
def split_neighbors_by_type(curr: str, neighbors: List[str]) -> Tuple[List[str], List[str]]:
    """
    把 curr 的邻居分成两类：
      - self_neighbors: 同一个原始 node 的其他 segment
      - inter_neighbors: 其他节点
    """
    curr_node, curr_seg = parse_token(curr)

    self_neighbors = []
    inter_neighbors = []

    for nbr in neighbors:
        nbr_node, nbr_seg = parse_token(nbr)
        if nbr_node == curr_node:
            self_neighbors.append(nbr)
        else:
            inter_neighbors.append(nbr)

    return self_neighbors, inter_neighbors


def choose_next_node_relax_rate(
    graph: nx.Graph,
    curr: str,
    relax_rate: float,
) -> str | None:
    """
    relax-rate 风格：
      1) 先按 relax_rate 决定这一步走 self 还是 interaction
      2) 再在该类别内部按边权归一化采样
    """
    neighbors = list(graph.neighbors(curr))
    if len(neighbors) == 0:
        return None

    self_neighbors, inter_neighbors = split_neighbors_by_type(curr, neighbors)

    has_self = len(self_neighbors) > 0
    has_inter = len(inter_neighbors) > 0

    # 决定这一步走哪一类
    if has_self and has_inter:
        go_self = np.random.rand() < relax_rate
        chosen_group = self_neighbors if go_self else inter_neighbors
    elif has_self:
        chosen_group = self_neighbors
    else:
        chosen_group = inter_neighbors

    weights = [graph[curr][nbr].get("weight", 1.0) for nbr in chosen_group]
    return weighted_choice(chosen_group, weights)


def relax_rate_walk(
    graph: nx.Graph,
    start_node: str,
    walk_length: int,
    relax_rate: float,
) -> List[str]:
    walk = [start_node]

    while len(walk) < walk_length:
        curr = walk[-1]
        nxt = choose_next_node_relax_rate(
            graph=graph,
            curr=curr,
            relax_rate=relax_rate,
        )
        if nxt is None:
            break
        walk.append(nxt)

    return walk


def generate_walks(
    graph: nx.Graph,
    num_walks: int,
    walk_length: int,
    relax_rate: float,
) -> List[List[str]]:
    nodes = list(graph.nodes())
    walks = []

    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walks.append(
                relax_rate_walk(
                    graph=graph,
                    start_node=node,
                    walk_length=walk_length,
                    relax_rate=relax_rate,
                )
            )

    return walks


print("Generating relax-rate walks...")
walks = generate_walks(
    graph=G,
    num_walks=NUM_WALKS,
    walk_length=WALK_LENGTH,
    relax_rate=RELAX_RATE,
)
print(f"Generated {len(walks)} walks.")


# =========================================================
# 10. 用 Word2Vec 训练 embedding
# =========================================================
print("Training Word2Vec on walks...")
w2v_model = Word2Vec(
    sentences=walks,
    vector_size=DIMENSIONS,
    window=WINDOW_SIZE,
    min_count=MIN_COUNT,
    sg=1,
    workers=WORKERS,
    batch_words=BATCH_WORDS,
    seed=RANDOM_SEED,
)

tokens = w2v_model.wv.index_to_key
emb = np.vstack([w2v_model.wv[tok] for tok in tokens]).astype(np.float32)

print("Embedding shape:", emb.shape)


# =========================================================
# 11. 保存 embedding 和映射
# =========================================================
np.save(EMB_NPY_OUT, emb)
print(f"Saved embedding matrix to: {EMB_NPY_OUT}")

emb_map_rows = []
for i, tok in enumerate(tokens):
    node, seg = parse_token(tok)
    emb_map_rows.append({
        "token": tok,
        "embedding_row": i,
        "node": node,
        "segment": seg
    })

emb_map_df = pd.DataFrame(emb_map_rows)
emb_map_df.to_csv(EMB_MAP_OUT, index=False)
print(f"Saved embedding map to: {EMB_MAP_OUT}")
print(emb_map_df.head())

Loaded 1965 rows from: segmented_by_event_jsd_cusum_l20.csv
Constructed 1530 interaction edges.
Saved node info to: node_segment_node_info_l20.csv
Constructed 159 self edges.
          u         v  weight    edge_type
0   0__seg0  37__seg0     1.0  interaction
1  48__seg0   5__seg0     4.0  interaction
2  28__seg0   7__seg0     1.0  interaction
3   0__seg0  40__seg0     1.0  interaction
4  10__seg0  12__seg0     4.0  interaction
Graph built: 209 nodes, 1689 edges
Generating relax-rate walks...
Generated 20900 walks.
Training Word2Vec on walks...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Embedding shape: (209, 64)
Saved embedding matrix to: node_segment_node2vec_l20.npy
Saved embedding map to: node_segment_node2vec_map_l20.csv
      token  embedding_row node  segment
0  23__seg1              0   23        1
1  25__seg1              1   25        1
2  23__seg0              2   23        0
3  25__seg0              3   25        0
4   1__seg2              4    1        2


In [102]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans


# =========================================================
# 1. 基本配置
# =========================================================
FILE_NAME = "p0.8_mu0.2_l20"
RESULTS_DIR = Path(f"results/{FILE_NAME}")

SEGMENTED_INPUT = "segmented_by_event_jsd_cusum_l20.csv"
EMB_NPY_PATH = "node_segment_node2vec_l20.npy"
EMB_MAP_PATH = "node_segment_node2vec_map_l20.csv"
OUT_PATH = "node_segment_node2vec_kmeans_l20.csv"

K = 5


# =========================================================
# 2. 读取 embedding 和映射
# =========================================================
emb = np.load(EMB_NPY_PATH).astype(np.float32)
emb_map_df = pd.read_csv(EMB_MAP_PATH)

required_emb_map_cols = {"token", "embedding_row", "node", "segment"}
missing = required_emb_map_cols - set(emb_map_df.columns)
if missing:
    raise ValueError(f"Missing columns in embedding map: {missing}")

if emb.shape[0] != len(emb_map_df):
    raise ValueError(
        f"Embedding row count mismatch: emb.shape[0]={emb.shape[0]}, "
        f"len(emb_map_df)={len(emb_map_df)}"
    )

print("Loaded embedding shape:", emb.shape)
print("Loaded embedding map rows:", len(emb_map_df))


# =========================================================
# 3. 聚类
# =========================================================
kmeans = KMeans(n_clusters=K, n_init=20, random_state=0)
labels = kmeans.fit_predict(emb)

emb_map_df = emb_map_df.copy()
emb_map_df["commu"] = labels

print("Cluster counts:")
print(pd.Series(labels).value_counts().sort_index())


# =========================================================
# 4. 构造 token -> cluster label 映射
# =========================================================
token_to_commu = dict(zip(emb_map_df["token"], emb_map_df["commu"]))


# =========================================================
# 5. 读取原始 segmented 文件，并还原 source/destination 的社区标签
# =========================================================
seg_df = pd.read_csv(SEGMENTED_INPUT)

required_seg_cols = {
    "source", "destination", "timestamp",
    "source_segment", "destination_segment"
}
missing = required_seg_cols - set(seg_df.columns)
if missing:
    raise ValueError(f"Missing columns in segmented input: {missing}")


def make_token(node, seg):
    return f"{node}__seg{seg}"


seg_df["source_token"] = seg_df.apply(
    lambda r: make_token(r["source"], r["source_segment"]), axis=1
)
seg_df["destination_token"] = seg_df.apply(
    lambda r: make_token(r["destination"], r["destination_segment"]), axis=1
)

seg_df["source_commu"] = seg_df["source_token"].map(token_to_commu)
seg_df["destination_commu"] = seg_df["destination_token"].map(token_to_commu)

# 检查是否有没映射上的
if seg_df["source_commu"].isna().any() or seg_df["destination_commu"].isna().any():
    bad_src = seg_df.loc[seg_df["source_commu"].isna(), "source_token"].unique().tolist()
    bad_dst = seg_df.loc[seg_df["destination_commu"].isna(), "destination_token"].unique().tolist()
    raise ValueError(
        "Some tokens cannot be mapped to cluster labels.\n"
        f"Bad source tokens (first 10): {bad_src[:10]}\n"
        f"Bad destination tokens (first 10): {bad_dst[:10]}"
    )

seg_df["source_commu"] = seg_df["source_commu"].astype(int)
seg_df["destination_commu"] = seg_df["destination_commu"].astype(int)


# =========================================================
# 6. 输出最终结果
# =========================================================
result_df = seg_df[[
    "source",
    "destination",
    "timestamp",
    "source_commu",
    "destination_commu"
]].copy()

result_df.to_csv(OUT_PATH, index=False)

print(f"Saved clustering result to: {OUT_PATH}")
print(result_df.head())

Loaded embedding shape: (209, 64)
Loaded embedding map rows: 209
Cluster counts:
0     24
1    108
2     28
3     24
4     25
Name: count, dtype: int64
Saved clustering result to: node_segment_node2vec_kmeans_l20.csv
   source  destination  timestamp  source_commu  destination_commu
0       0           37          6             1                  4
1       5           48          9             1                  1
2       7           28         16             2                  3
3       0           40         20             1                  1
4      10           12         24             1                  1


In [96]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture


# =========================================================
# 1. 基本配置
# =========================================================
FILE_NAME = "p0.8_mu0.2_l20"
RESULTS_DIR = Path(f"results/{FILE_NAME}")

SEGMENTED_INPUT = "segmented_by_event_jsd_cusum_l20.csv"
EMB_NPY_PATH = "node_segment_node2vec_l20.npy"
EMB_MAP_PATH = "node_segment_node2vec_map_l20.csv"

OUT_PATH = "node_segment_node2vec_gmm_l20.csv"

K = 5


# =========================================================
# 2. 读取 embedding 和映射
# =========================================================
emb = np.load(EMB_NPY_PATH).astype(np.float32)
emb_map_df = pd.read_csv(EMB_MAP_PATH)

required_emb_map_cols = {"token", "embedding_row", "node", "segment"}
missing = required_emb_map_cols - set(emb_map_df.columns)
if missing:
    raise ValueError(f"Missing columns in embedding map: {missing}")

if emb.shape[0] != len(emb_map_df):
    raise ValueError(
        f"Embedding row count mismatch: emb.shape[0]={emb.shape[0]}, "
        f"len(emb_map_df)={len(emb_map_df)}"
    )

print("Loaded embedding shape:", emb.shape)
print("Loaded embedding map rows:", len(emb_map_df))


# =========================================================
# 3. GMM 聚类
# =========================================================
gmm = GaussianMixture(
    n_components=K,
    covariance_type="full",   # 可改成 "diag" / "tied" / "spherical"
    n_init=10,
    random_state=0
)

labels = gmm.fit_predict(emb)
probs = gmm.predict_proba(emb)   # [N, K]

emb_map_df = emb_map_df.copy()
emb_map_df["commu"] = labels
emb_map_df["confidence"] = probs.max(axis=1)

print("Cluster counts:")
print(pd.Series(labels).value_counts().sort_index())

print("Mean confidence:", probs.max(axis=1).mean())


# =========================================================
# 4. 构造 token -> cluster label 映射
# =========================================================
token_to_commu = dict(zip(emb_map_df["token"], emb_map_df["commu"]))
token_to_conf = dict(zip(emb_map_df["token"], emb_map_df["confidence"]))


# =========================================================
# 5. 读取原始 segmented 文件，并还原 source/destination 的社区标签
# =========================================================
seg_df = pd.read_csv(SEGMENTED_INPUT)

required_seg_cols = {
    "source", "destination", "timestamp",
    "source_segment", "destination_segment"
}
missing = required_seg_cols - set(seg_df.columns)
if missing:
    raise ValueError(f"Missing columns in segmented input: {missing}")


def make_token(node, seg):
    return f"{node}__seg{seg}"


seg_df["source_token"] = seg_df.apply(
    lambda r: make_token(r["source"], r["source_segment"]), axis=1
)
seg_df["destination_token"] = seg_df.apply(
    lambda r: make_token(r["destination"], r["destination_segment"]), axis=1
)

seg_df["source_commu"] = seg_df["source_token"].map(token_to_commu)
seg_df["destination_commu"] = seg_df["destination_token"].map(token_to_commu)

seg_df["source_confidence"] = seg_df["source_token"].map(token_to_conf)
seg_df["destination_confidence"] = seg_df["destination_token"].map(token_to_conf)

# 检查是否有没映射上的
if seg_df["source_commu"].isna().any() or seg_df["destination_commu"].isna().any():
    bad_src = seg_df.loc[seg_df["source_commu"].isna(), "source_token"].unique().tolist()
    bad_dst = seg_df.loc[seg_df["destination_commu"].isna(), "destination_token"].unique().tolist()
    raise ValueError(
        "Some tokens cannot be mapped to cluster labels.\n"
        f"Bad source tokens (first 10): {bad_src[:10]}\n"
        f"Bad destination tokens (first 10): {bad_dst[:10]}"
    )

seg_df["source_commu"] = seg_df["source_commu"].astype(int)
seg_df["destination_commu"] = seg_df["destination_commu"].astype(int)


# =========================================================
# 6. 输出最终结果
# =========================================================
result_df = seg_df[[
    "source",
    "destination",
    "timestamp",
    "source_commu",
    "destination_commu",
    "source_confidence",
    "destination_confidence"
]].copy()

result_df.to_csv(OUT_PATH, index=False)

print(f"Saved clustering result to: {OUT_PATH}")
print(result_df.head())

Loaded embedding shape: (288, 64)
Loaded embedding map rows: 288
Cluster counts:
0     33
1     54
2    142
3     28
4     31
Name: count, dtype: int64
Mean confidence: 1.0
Saved clustering result to: node_segment_node2vec_gmm_l20.csv
   source  destination  timestamp  source_commu  destination_commu  \
0       0           37          6             2                  2   
1       5           48          9             2                  2   
2       7           28         16             2                  2   
3       0           40         20             2                  2   
4      10           12         24             2                  2   

   source_confidence  destination_confidence  
0                1.0                     1.0  
1                1.0                     1.0  
2                1.0                     1.0  
3                1.0                     1.0  
4                1.0                     1.0  
